# Phase 3 — Circuit Sharing: Addition vs Multiplication (Direct Comparison)

## Why this design

Phase 2 (3-task: add+sub+mul) showed that **three tasks simultaneously exceeded this model's capacity** — none grokked. That is itself a finding (capacity collapse), but it prevented us from doing clean circuit analysis.

Phase 3 uses a **direct 2-task comparison** strategy:

| | Phase 1 (done) | Phase 3 (this notebook) |
|---|---|---|
| Tasks | add + sub | add + mul |
| Algebraic relationship | Very close (sub = add with negation) | Distant (mul uses discrete log, not rotation) |
| Expected sharing | HIGH — same Fourier circuit | LOWER — different algorithmic strategy |
| Grokking | Same epoch (6200) | Expected: different epochs for each task |

By comparing **the same metrics** (grokking timing, Fourier frequencies, head ablation, activation patching) between Phase 1 and Phase 3, we directly measure how algebraic structure affects circuit reuse.

## What success looks like

| Metric | Phase 1 result | Phase 3 expectation | What it proves |
|--------|---------------|--------------------|-----------------|
| Grokking timing | Same epoch (6200) for both | Different epochs for add vs mul | Separate learning events → separate circuits |
| Dominant Fourier freqs | {9, 47, 49} — same 3 for both | Different frequencies for add vs mul | Different representations used |
| Head ablation | (Phase 1 invalid; Phase 3 is first valid data) | Some heads mul-specific, some add-specific | Task-specific heads identified |
| add↔mul patching | N/A | Lower than Phase 1's add↔sub (15-22%) | Quantifies reduced sharing |

## Cell 1 — Install

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformer_lens>=1.0,<3', 'einops', 'tqdm'])
print('Done. Runtime -> Restart session -> run from Cell 2.')

## Cell 2 — Imports

In [ ]:
import os, random, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import trange
from transformer_lens import HookedTransformer, HookedTransformerConfig
from importlib.metadata import version as _pkg_version
try:
    _tl_ver = _pkg_version('transformer_lens')
except Exception:
    import transformer_lens as _tl
    _tl_ver = getattr(_tl, '__version__', 'unknown')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'OK | transformer_lens {_tl_ver} | {DEVICE}')

## Cell 3 — Hyperparameters

**Same as Phase 1** (which successfully grokked). Only the ops tuple changes.

In [ ]:
P          = 113
OPS        = ('add', 'mul')   # Phase 3: add vs mul — the algebraic contrast
TRAIN_FRAC = 0.3              # same as Phase 1
D_MODEL, N_HEADS, D_MLP, N_LAYERS, SEQ_LEN = 128, 4, 512, 1, 4
EPOCHS, LR, WEIGHT_DECAY = 20_000, 1e-3, 1.0   # same regime as Phase 1
CHECKPOINT_EVERY = 200

# Phase 1 reference values (hardcoded from your Phase 1 run)
PHASE1_GROK_EPOCH    = 6200
PHASE1_FOURIER_FREQS = {9, 47, 49}
PHASE1_PATCH_OVERLAP = 0.185  # average of add->sub (0.15) and sub->add (0.22)

os.makedirs('results', exist_ok=True)
print(f'Hyperparameters set. ops={OPS}, WD={WEIGHT_DECAY}, epochs={EPOCHS}')
print(f'Phase 1 reference: grok@{PHASE1_GROK_EPOCH}, freqs={PHASE1_FOURIER_FREQS}, patch={PHASE1_PATCH_OVERLAP:.3f}')

## Cell 4 — Data

In [ ]:
def make_dataset(p=113, ops=('add', 'mul'), train_frac=0.3, seed=0):
    OP_FNS = {
        'add': lambda a, b, p: (a + b) % p,
        'sub': lambda a, b, p: (a - b) % p,
        'mul': lambda a, b, p: (a * b) % p,
    }
    rng = np.random.RandomState(seed)
    op_list = list(ops)
    equals_token = p + len(op_list)
    all_inputs, all_labels = [], []
    for op_idx, op_name in enumerate(op_list):
        op_token = p + op_idx
        fn = OP_FNS[op_name]
        for a in range(p):
            for b in range(p):
                all_inputs.append([a, op_token, b, equals_token])
                all_labels.append(fn(a, b, p))
    inputs = torch.tensor(all_inputs, dtype=torch.long)
    labels = torch.tensor(all_labels, dtype=torch.long)
    n = len(inputs)
    perm = rng.permutation(n)
    n_train = int(train_frac * n)
    train_data = (inputs[perm[:n_train]], labels[perm[:n_train]])
    test_data  = (inputs[perm[n_train:]], labels[perm[n_train:]])
    return train_data, test_data, p + len(op_list) + 1, equals_token, op_list

train_data, test_data, VOCAB_SIZE, EQUALS_TOKEN, OP_LIST = make_dataset(
    p=P, ops=OPS, train_frac=TRAIN_FRAC, seed=SEED
)
OP_TOKENS = {op: P + i for i, op in enumerate(OP_LIST)}
print(f'Dataset: vocab={VOCAB_SIZE}, equals={EQUALS_TOKEN}, op_tokens={OP_TOKENS}')
print(f'Train: {train_data[0].shape[0]:,} | Test: {test_data[0].shape[0]:,}')

## Cell 5 — Model (with `use_attn_result=True`)

In [ ]:
def build_model(vocab_size, d_model=128, n_heads=4, d_mlp=512, n_layers=1,
                seq_len=4, device='cuda', seed=0):
    cfg = HookedTransformerConfig(
        n_layers=n_layers, d_model=d_model, d_head=d_model // n_heads,
        n_heads=n_heads, d_mlp=d_mlp, d_vocab=vocab_size, n_ctx=seq_len,
        act_fn='relu', normalization_type=None,
        use_attn_result=True,  # Required for valid head ablation
        seed=seed, device=device,
    )
    return HookedTransformer(cfg)

model = build_model(vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS,
                    d_mlp=D_MLP, n_layers=N_LAYERS, seq_len=SEQ_LEN,
                    device=DEVICE, seed=SEED)
n_params = sum(p.numel() for p in model.parameters())
dummy = torch.zeros(1, SEQ_LEN, dtype=torch.long).to(DEVICE)
_, cache = model.run_with_cache(dummy)
print(f'Model: {n_params:,} params | hook_result available: {"blocks.0.attn.hook_result" in cache}')

## Cell 6 — Per-Task Accuracy Helper

In [ ]:
@torch.no_grad()
def per_task_accuracy(model, inputs, labels, op_tokens_map, device=DEVICE):
    model.eval()
    inputs, labels = inputs.to(device), labels.to(device)
    logits = model(inputs)[:, -1, :]
    preds  = logits.argmax(-1)
    return {
        op: (preds[inputs[:, 1] == tok] == labels[inputs[:, 1] == tok]).float().mean().item()
        if (inputs[:, 1] == tok).sum() > 0 else float('nan')
        for op, tok in op_tokens_map.items()
    }
print('per_task_accuracy() ready.')

## Cell 7 — Training

**Expected runtime:** ~12–15 min on T4.  
**Watch for:** Add and mul grokking at **different** epochs — unlike Phase 1 where both grokked together at 6200. A gap between grokking epochs is evidence of separate learning dynamics.

In [ ]:
train_inputs, train_labels = [x.to(DEVICE) for x in train_data]
test_inputs,  test_labels  = [x.to(DEVICE) for x in test_data]
optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                               weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))

history = {'epoch': [], 'train_loss': [], 'test_loss': []}
for op in OP_LIST:
    history[f'train_acc_{op}'] = []
    history[f'test_acc_{op}']  = []

for epoch in trange(EPOCHS, desc='Training'):
    model.train()
    logits = model(train_inputs)[:, -1, :]
    loss   = F.cross_entropy(logits, train_labels)
    loss.backward(); optimizer.step(); optimizer.zero_grad()

    if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            tr_l = F.cross_entropy(model(train_inputs)[:, -1, :], train_labels).item()
            te_l = F.cross_entropy(model(test_inputs)[:, -1, :],  test_labels).item()
        tr_task = per_task_accuracy(model, train_inputs, train_labels, OP_TOKENS)
        te_task = per_task_accuracy(model, test_inputs,  test_labels,  OP_TOKENS)
        history['epoch'].append(epoch)
        history['train_loss'].append(tr_l); history['test_loss'].append(te_l)
        for op in OP_LIST:
            history[f'train_acc_{op}'].append(tr_task[op])
            history[f'test_acc_{op}'].append(te_task[op])

print('\nTraining complete.')
grok_epochs = {}
for op in OP_LIST:
    acc = history[f'test_acc_{op}'][-1]
    ep  = next((e for e, a in zip(history['epoch'], history[f'test_acc_{op}']) if a > 0.95), 'N/A')
    status = 'GROKKED' if acc >= 0.95 else 'DID NOT GROK'
    print(f'  {op}: acc={acc:.4f}  {status}  grok_epoch={ep}')
    if acc >= 0.95: grok_epochs[op] = ep

## Cell 8 — Grokking Curves + Phase 1 Comparison

**The key diagnostic:** Did add and mul grok at the same epoch (shared dynamics) or different epochs (separate circuits)? Compare the grokking timing to Phase 1 where both tasks grokked simultaneously at epoch 6200.

Saves `results/phase3_grokking_curves.png`

In [ ]:
COLORS = {'add': 'steelblue', 'mul': 'seagreen'}
epochs = history['epoch']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'Phase 3 — Add + Mul (mod {P}) | Direct comparison to Phase 1 (Add + Sub)',
             fontsize=13, fontweight='bold')

# Loss
ax = axes[0]
ax.semilogy(epochs, history['train_loss'], label='Train loss', color='steelblue')
ax.semilogy(epochs, history['test_loss'],  label='Test loss',  color='tomato')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log)'); ax.set_title('Loss Curves')
ax.legend(); ax.grid(True, which='both', alpha=0.3)

# Per-task accuracy
ax = axes[1]
for op in OP_LIST:
    c = COLORS[op]
    ax.plot(epochs, history[f'train_acc_{op}'], '--', color=c, alpha=0.4, label=f'Train [{op}]')
    ax.plot(epochs, history[f'test_acc_{op}'],  '-',  color=c,             label=f'Test  [{op}]')
    ep = grok_epochs.get(op)
    if ep:
        ax.axvline(ep, linestyle=':', color=c, alpha=0.8)
        ax.text(ep+100, 0.05, f'{op}\n@{ep}', color=c, fontsize=8)

# Phase 1 reference line
ax.axvline(PHASE1_GROK_EPOCH, linestyle='--', color='gray', alpha=0.5, linewidth=1.5,
           label=f'Phase 1 grok (add+sub @ {PHASE1_GROK_EPOCH})')

ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Per-Task Accuracy vs Phase 1 Reference')
ax.set_ylim(-0.05, 1.08); ax.legend(loc='upper left', fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/phase3_grokking_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/phase3_grokking_curves.png')

# KEY COMPARISON
print(f'\n--- GROKKING TIMING COMPARISON ---')
print(f'Phase 1 (add+sub): BOTH grokked at epoch {PHASE1_GROK_EPOCH} (simultaneous)')
print(f'Phase 3 (add+mul):', end='')
if len(grok_epochs) == 2:
    diff = abs(grok_epochs['add'] - grok_epochs['mul'])
    print(f' add@{grok_epochs["add"]}, mul@{grok_epochs["mul"]} (gap={diff} epochs)')
    if diff > 1000:
        print('  -> SEPARATE learning events -> evidence of DIFFERENT circuits')
    else:
        print('  -> Near-simultaneous -> possible shared circuit even for mul')
elif len(grok_epochs) == 1:
    op = list(grok_epochs.keys())[0]
    missing = [o for o in OP_LIST if o not in grok_epochs][0]
    print(f' only {op} grokked ({grok_epochs[op]}); {missing} did not')
    print(f'  -> {missing} requires FUNDAMENTALLY different capacity than {op}')
else:
    print(' neither grokked — try EPOCHS=25000 or WEIGHT_DECAY=0.8')

## Cell 9 — Fourier Analysis vs Phase 1

**Key question:** Does the add+mul model use the same Fourier frequencies as Phase 1 (add+sub)?  
- Same frequencies → same circuit being reused  
- Different frequencies → mul forces different representations

Saves `results/phase3_fourier_spectrum.png`

In [ ]:
def fourier_embedding_analysis(model, p):
    W_E = model.W_E[:p].detach().cpu().numpy()
    power = (np.abs(np.fft.rfft(W_E, axis=0)) ** 2).sum(axis=1)
    return power

power = fourier_embedding_analysis(model, P)
freqs = np.arange(len(power))
top_k = 6
top_idx = np.argsort(power)[-top_k:][::-1]
top_nondc = sorted([int(f) for f in top_idx if f > 0])

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(freqs, power, color='steelblue', edgecolor='white', linewidth=0.3)
for idx in top_idx:
    ax.bar(idx, power[idx], color='crimson')
    ax.annotate(f'f={idx}', xy=(idx, power[idx]),
                xytext=(idx + 0.8, power[idx] * 1.04),
                fontsize=8, color='crimson',
                arrowprops=dict(arrowstyle='->', color='crimson', lw=0.7))

# Mark Phase 1 freqs in green outline
for f in PHASE1_FOURIER_FREQS:
    if f < len(power):
        ax.bar(f, power[f], color='none', edgecolor='lime', linewidth=2.5,
               label='Phase 1 freq' if f == list(PHASE1_FOURIER_FREQS)[0] else '')

ax.set_xlabel('Frequency')
ax.set_ylabel('Power (sum over d_model)')
ax.set_title(
    f'Phase 3 — Fourier Spectrum: Add + Mul (mod {P})\n'
    f'Red = top-{top_k} | Green outline = Phase 1 frequencies {sorted(PHASE1_FOURIER_FREQS)}'
)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/phase3_fourier_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

overlap = set(top_nondc) & PHASE1_FOURIER_FREQS
new_f   = set(top_nondc) - PHASE1_FOURIER_FREQS
print(f'Phase 3 top non-DC frequencies: {top_nondc}')
print(f'Phase 1 top frequencies:        {sorted(PHASE1_FOURIER_FREQS)}')
print(f'Overlap: {sorted(overlap)} | New in Phase 3: {sorted(new_f)}')
if len(overlap) >= 2:
    print('  -> Strong frequency overlap with Phase 1 -> circuit partially shared')
elif len(overlap) == 0:
    print('  -> Zero overlap with Phase 1 -> completely different circuit for mul')
else:
    print('  -> Partial overlap -> some shared, some new')

## Cell 10 — Head Ablation Sweep

First **valid** head ablation in the project (Phase 1's was broken; Phase 2's had uninterpretable baselines).  
**Both tasks grokked here**, so accuracy drops are meaningful.

| Pattern | Meaning |
|---------|--------|
| Head drops both add AND mul | Shared head |
| Head drops only add (mul fine) | Add-specific head |
| Head drops only mul (add fine) | Mul-specific head |
| Head drops neither | Not critical |

Compare to Phase 1: if add+sub shared ALL heads, but add+mul have some task-specific heads → algebraic structure drives head specialisation.

Saves `results/phase3_head_ablation.png`

In [ ]:
@torch.no_grad()
def ablate_head_accuracy(model, inputs, labels, op_tokens_map,
                          layer=0, head_idx=0, device=DEVICE):
    model.eval()
    inputs, labels = inputs.to(device), labels.to(device)
    hook_name = f'blocks.{layer}.attn.hook_result'
    def zero_head(act, hook):
        a = act.clone(); a[:, :, head_idx, :] = 0.0; return a
    logits = model.run_with_hooks(inputs, fwd_hooks=[(hook_name, zero_head)])[:, -1, :]
    preds  = logits.argmax(-1)
    return {
        op: (preds[inputs[:, 1] == tok] == labels[inputs[:, 1] == tok]).float().mean().item()
        if (inputs[:, 1] == tok).sum() > 0 else float('nan')
        for op, tok in op_tokens_map.items()
    }

baseline = per_task_accuracy(model, test_inputs, test_labels, OP_TOKENS)
print('Baseline test accuracy (no ablation):')
for op, acc in baseline.items(): print(f'  {op}: {acc:.4f}')

THRESHOLD = 0.10
ablation = {}
for h in range(N_HEADS):
    ablated = ablate_head_accuracy(model, test_inputs, test_labels, OP_TOKENS, head_idx=h)
    ablation[h] = {op: baseline[op] - ablated[op] for op in OP_LIST}
    drops_str = '  '.join(f'{op}: {ablation[h][op]:+.4f}' for op in OP_LIST)
    print(f'  Head {h}: {drops_str}')
print('\nAblation done.')

In [ ]:
x, bar_w = np.arange(N_HEADS), 0.35
fig, ax = plt.subplots(figsize=(10, 5))
for i, op in enumerate(OP_LIST):
    drops  = [ablation[h][op] for h in range(N_HEADS)]
    offset = (i - len(OP_LIST)/2 + 0.5) * bar_w
    bars = ax.bar(x + offset, drops, bar_w, label=f'drop [{op}]',
                  color=COLORS[op], edgecolor='white')
    for bar, val in zip(bars, drops):
        if not math.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax.axhline(THRESHOLD, color='red', linestyle='--', linewidth=0.8, label='10% threshold')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels([f'Head {h}' for h in range(N_HEADS)])
ax.set_ylabel('Accuracy drop (baseline - ablated)')
ax.set_title(
    f'Phase 3 — Head Ablation: Add vs Mul (mod {P})\n'
    'FIRST valid head ablation in this project (use_attn_result=True, both tasks grokked)'
)
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(-0.05, max(0.5, max(ablation[h][op] for h in range(N_HEADS) for op in OP_LIST) + 0.15))
plt.tight_layout()
plt.savefig('results/phase3_head_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/phase3_head_ablation.png')

print('\nInterpretation:')
shared_heads, add_heads, mul_heads, none_heads = [], [], [], []
for h in range(N_HEADS):
    crit = [op for op in OP_LIST if ablation[h][op] >= THRESHOLD]
    if set(crit) == set(OP_LIST): shared_heads.append(h)
    elif crit == ['add']:         add_heads.append(h)
    elif crit == ['mul']:         mul_heads.append(h)
    else:                         none_heads.append(h)
    label = ('SHARED' if set(crit)==set(OP_LIST) else
             f'TASK-SPECIFIC [{crit[0]}]' if len(crit)==1 else
             'NOT CRITICAL' if not crit else f'PARTIAL {crit}')
    print(f'  Head {h}: {label}')
print(f'\nSummary: shared={shared_heads}, add-specific={add_heads}, mul-specific={mul_heads}, neither={none_heads}')

## Cell 11 — Activation Patching: Add ↔ Mul

**The central quantitative comparison:**  
Phase 1 (add↔sub) patching overlap: ~15–22%  
Phase 3 (add↔mul) patching overlap: expected to be LOWER

The difference between these two numbers is your primary empirical result.

Saves `results/phase3_activation_patching.png`

In [ ]:
FIXED_B = 7
OP_FNS_MATCHED = {'add': lambda a: (a + FIXED_B) % P,
                   'mul': lambda a: (a * FIXED_B) % P}

op_inputs, op_labels = {}, {}
for op in OP_LIST:
    tok = OP_TOKENS[op]
    op_inputs[op] = torch.tensor([[a, tok, FIXED_B, EQUALS_TOKEN] for a in range(P)],
                                  dtype=torch.long).to(DEVICE)
    op_labels[op] = torch.tensor([OP_FNS_MATCHED[op](a) for a in range(P)],
                                  dtype=torch.long).to(DEVICE)

@torch.no_grad()
def patch_accuracy(model, clean_op, corrupted_op, hook_name):
    try:
        _, clean_cache = model.run_with_cache(op_inputs[clean_op])
        clean_act = clean_cache[hook_name]
        def patch_fn(act, hook):
            out = act.clone()
            b = min(out.shape[0], clean_act.shape[0])
            out[:b] = clean_act[:b]; return out
        logits = model.run_with_hooks(op_inputs[corrupted_op],
                                       fwd_hooks=[(hook_name, patch_fn)])[:, -1, :]
        return (logits.argmax(-1) == op_labels[corrupted_op]).float().mean().item()
    except Exception as e:
        print(f'  Hook {hook_name} failed: {e}'); return float('nan')

baselines = {}
for op in OP_LIST:
    model.eval()
    with torch.no_grad():
        logits = model(op_inputs[op])[:, -1, :]
        baselines[op] = (logits.argmax(-1) == op_labels[op]).float().mean().item()
print('Baselines:', {op: f'{v:.4f}' for op, v in baselines.items()})

HOOK_POINTS = [
    ('hook_embed',              'Token\nEmbed'),
    ('blocks.0.attn.hook_result', 'Attn\nOutput'),
    ('blocks.0.hook_mlp_out',   'MLP\nOutput'),
]

patch_results = {}
for hn, hl in HOOK_POINTS:
    a2m = patch_accuracy(model, 'add', 'mul', hn)
    m2a = patch_accuracy(model, 'mul', 'add', hn)
    patch_results[hl] = {'add->mul': a2m, 'mul->add': m2a}
    print(f'  [{hl.replace(chr(10)," "):12s}]  add->mul: {a2m:.4f}  |  mul->add: {m2a:.4f}')

print('\nPatching done.')

In [ ]:
hook_labels = list(patch_results.keys())
a2m_vals = [patch_results[h]['add->mul'] for h in hook_labels]
m2a_vals = [patch_results[h]['mul->add'] for h in hook_labels]
x, bar_w = np.arange(len(hook_labels)), 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - bar_w/2, a2m_vals, bar_w, label='add->mul patch (mul acc)', color='steelblue')
bars2 = ax.bar(x + bar_w/2, m2a_vals, bar_w, label='mul->add patch (add acc)', color='seagreen')

# Phase 1 reference band
ax.axhspan(0.15, 0.22, alpha=0.12, color='orange',
           label=f'Phase 1 add<->sub range (0.15-0.22)')

for bar, val in zip(list(bars1)+list(bars2), a2m_vals+m2a_vals):
    if not math.isnan(val):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(hook_labels)
ax.set_ylabel('Accuracy after patching')
ax.set_ylim(0, 1.2)
ax.set_title(
    f'Phase 3 — Activation Patching: Add ↔ Mul (mod {P}, b={FIXED_B})\n'
    f'Orange band = Phase 1 (add↔sub) overlap range for comparison'
)
ax.legend(fontsize=8); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/phase3_activation_patching.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/phase3_activation_patching.png')

# KEY NUMBER
phase3_overlap = np.nanmean([v for h in hook_labels for v in patch_results[h].values()])
print(f'\n--- THE CENTRAL QUANTITATIVE RESULT ---')
print(f'Phase 1 (add<->sub) mean patching overlap: {PHASE1_PATCH_OVERLAP:.3f} (18.5%)')
print(f'Phase 3 (add<->mul) mean patching overlap: {phase3_overlap:.3f} ({phase3_overlap*100:.1f}%)')
reduction = (PHASE1_PATCH_OVERLAP - phase3_overlap) / PHASE1_PATCH_OVERLAP * 100
print(f'Reduction in sharing: {reduction:.1f}%')
print(f'\nThis is the number to put in your report.')

## Cell 12 — Final Summary Dashboard

Everything in one place. Copy this into `docs/proposal.md`.

In [ ]:
print('=' * 72)
print('PHASE 3 SUMMARY — Add + Mul (mod 113)')
print('Direct comparison to Phase 1 (Add + Sub)')
print('=' * 72)

print('\n1. GROKKING TIMING')
print(f'   Phase 1 (add+sub): both grokked simultaneously at epoch {PHASE1_GROK_EPOCH}')
print(f'   Phase 3 (add+mul):')
for op in OP_LIST:
    acc = history[f'test_acc_{op}'][-1]
    ep = grok_epochs.get(op, 'DID NOT GROK')
    print(f'     {op}: acc={acc:.4f}, grok_epoch={ep}')
if len(grok_epochs) == 2:
    diff = abs(grok_epochs['add'] - grok_epochs['mul'])
    print(f'   Grokking gap: {diff} epochs (Phase 1 gap was 0)')

print('\n2. FOURIER FREQUENCIES')
print(f'   Phase 1: {sorted(PHASE1_FOURIER_FREQS)}')
print(f'   Phase 3: {top_nondc}')
print(f'   Overlap: {sorted(overlap)}')

print('\n3. HEAD ABLATION (first valid measurement)')
print(f'   Shared heads: {shared_heads}')
print(f'   Add-specific: {add_heads}')
print(f'   Mul-specific: {mul_heads}')
print(f'   Not critical: {none_heads}')

print('\n4. ACTIVATION PATCHING')
print(f'   Phase 1 (add<->sub): {PHASE1_PATCH_OVERLAP:.3f} mean overlap')
print(f'   Phase 3 (add<->mul): {phase3_overlap:.3f} mean overlap')
print(f'   Reduction: {reduction:.1f}%')
for hl, v in patch_results.items():
    lbl = hl.replace(chr(10), ' ')
    print(f'   {lbl}: add->mul={v["add->mul"]:.4f}, mul->add={v["mul->add"]:.4f}')

print('\n5. SAVED FILES')
for f in ['results/phase3_grokking_curves.png', 'results/phase3_fourier_spectrum.png',
          'results/phase3_head_ablation.png', 'results/phase3_activation_patching.png']:
    print(f'   [{"OK" if os.path.exists(f) else "MISSING"}] {f}')

print('\n' + '=' * 72)
print('NEXT: download 4 PNGs, commit to results/, git push')
print('This completes Phase 3. The project now has a complete arc:')
print('  Phase 1: add+sub share circuit (high overlap, simultaneous grokking)')
print('  Phase 2: 3-task add+sub+mul causes capacity collapse')
print('  Phase 3: add+mul share LESS circuit than add+sub (algebraic structure matters)')
print('=' * 72)

## Cell 13 — (Optional) Save to Drive

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = '/content/drive/MyDrive/mech_interp_phase3'
    os.makedirs(save_dir, exist_ok=True)
    torch.save(model.state_dict(), f'{save_dir}/model_final.pt')
    import json, shutil
    with open(f'{save_dir}/history.json', 'w') as f: json.dump(history, f)
    for png in ['phase3_grokking_curves.png','phase3_fourier_spectrum.png',
                'phase3_head_ablation.png','phase3_activation_patching.png']:
        shutil.copy(f'results/{png}', f'{save_dir}/{png}')
    print(f'Saved to {save_dir}')
else:
    print('Set SAVE_TO_DRIVE=True to persist. Remember to download PNGs before session ends!')